# 📓 Notebook 03 — Mobile (Android / iOS)

Static APK analysis, hardcoded secrets, root/SSL-pinning bypass, exported components, dynamic instrumentation.

# 🧱 BASE (shared setup) — run these first, in order

In [ ]:
# Which GPU did you actually get? Decide the model from this.
!nvidia-smi
import torch, psutil
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM {p.total_memory/1e9:.1f} GB | RAM {psutil.virtual_memory().total/1e9:.1f} GB")
!df -h /content | tail -1

In [ ]:
import os
try:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("cache:", CACHE)
except Exception as e:
    print("Drive notice:", e)
    CACHE = '/content/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("local cache:", CACHE)


In [ ]:
%%bash
apt-get -qq update
apt-get -qq install -y docker.io git curl wget jq unzip tmux nodejs npm python3-venv python3-pip net-tools iputils-ping dnsutils
# Colab blocks Docker's iptables hooks -> disable them
dockerd --iptables=false --ip6tables=false --bridge=none >/var/log/dockerd.log 2>&1 &
for i in $(seq 1 30); do docker info >/dev/null 2>&1 && break; sleep 2; done
docker info | grep -E "Server Version|Storage Driver"
echo "docker ready" 

In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh || true
command -v ollama || { curl -L https://ollama.com/download/ollama-linux-amd64.tgz | tar -xz -C /usr; }
nohup ollama serve >/var/log/ollama.log 2>&1 &
for i in $(seq 1 40); do curl -s localhost:11434/api/version >/dev/null && break; sleep 2; done
curl -s localhost:11434/api/version && echo " <- ollama up" 

In [ ]:
%%bash
if ollama list | grep -q "qwen2.5-coder:7b"; then
  echo "qwen2.5-coder:7b ready"
else
  ollama pull qwen2.5-coder:7b
fi
ollama list


In [ ]:
%%bash
# HexStrike = 150+ tools behind one MCP endpoint
if [ ! -d /content/hexstrike ]; then
  git clone --depth 1 https://github.com/0x4m4/hexstrike-ai /content/hexstrike
fi
cd /content/hexstrike
if ! curl -sf localhost:8888/health >/dev/null; then
  nohup python3 hexstrike_server.py --port 8888 >/var/log/hexstrike.log 2>&1 &
  sleep 5
fi
echo "hexstrike:"; curl -s localhost:8888/health; echo

# Kali toolbox container the agent can shell into
docker pull -q kalilinux/kali-rolling || true
docker run -d --name redteam --network host kalilinux/kali-rolling sleep infinity || true
echo toolbox-ready


In [ ]:
%%bash
if ! pgrep -f "cloudflared tunnel --url http://localhost:8888" >/dev/null; then
  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared 2>/dev/null || true
  chmod +x /usr/local/bin/cloudflared 2>/dev/null || true
  nohup cloudflared tunnel --url http://localhost:8888 >/var/log/cf.log 2>&1 &
  sleep 10
fi
echo "HexStrike URL:"; grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /var/log/cf.log | head -1


In [ ]:
%%bash
apt-get -qq install -y default-jdk android-tools-adb apktool aapt unzip
pip -q install frida-tools objection apkleaks 2>/dev/null || true
# jadx
JADX=1.5.0
wget -q https://github.com/skylot/jadx/releases/download/v${JADX}/jadx-${JADX}.zip -O /tmp/jadx.zip
unzip -q -o /tmp/jadx.zip -d /opt/jadx && ln -sf /opt/jadx/bin/jadx /usr/local/bin/jadx
# MobSF (web UI on :8000)
docker pull -q opensecurity/mobile-security-framework-mobsf
echo "mobile tools ready" 

In [ ]:
%%bash
mkdir -p /content/apks
if [ ! -d /content/apks/diva ]; then
  git clone --depth 1 https://github.com/payatu/diva-android /content/apks/diva 2>/dev/null || true
fi
if [ ! -d /content/apks/AndroGoat ]; then
  git clone --depth 1 https://github.com/satishpatnayak/AndroGoat /content/apks/AndroGoat 2>/dev/null || true
fi
find /content/apks -name "*.apk" | head


In [ ]:
%%bash
docker run -d --name mobsf --network host opensecurity/mobile-security-framework-mobsf
sleep 25
echo "MobSF -> http://localhost:8000  (user: mobsf / pass: mobsf)"
curl -s -o /dev/null -w "mobsf:%{http_code}\n" http://localhost:8000

In [ ]:
# --- static analysis without a device ---
import glob, os
apks = glob.glob('/content/apks/**/*.apk', recursive=True)
apk = apks[0] if apks else '/content/apks/diva-beta.apk'
print(f'Analyzing APK: {apk}')
!jadx -d /content/diva_src {apk} 2>/dev/null || true
!apktool d -f -o /content/diva_apktool {apk} 2>/dev/null || true
!grep -rEi "api[_-]?key|secret|password|token" /content/diva_src 2>/dev/null | head -20 || true
!strings {apk} 2>/dev/null | grep -Ei "http|firebase|amazonaws" | sort -u | head || true


In [ ]:
%%bash
# on the VM
nohup frida-server >/dev/null 2>&1 &   # or push frida-server to device
echo "adb devices"; adb devices || true
echo "frida-ps -U"; frida-ps -U 2>/dev/null | head || true